# Интеллектуальная система генерации стихов

Финальный Colab-блокнот для учебного проекта.

В проекте есть два подхода:

1. Цепи Маркова.
2. LSTM с простым механизмом attention.

Блокнот рассчитан на запуск сверху вниз. Сначала мы подготавливаем проект и датасет, потом запускаем Markov-часть, затем LSTM-часть.

## 1. Настройки

Если проект лежит в другом репозитории, поменяйте `REPO_URL`.

In [ ]:
import os

REPO_URL = "https://github.com/morganizwd/poetry.git"
PROJECT_DIR = "/content/poetry"

DATASET_JSON_URL = "https://raw.githubusercontent.com/Koziev/Rifma/main/rifma_dataset.json"
DATASET_JSON_PATH = "/content/rifma_dataset.json"
DATASET_TXT_PATH = "/content/poems_clean.txt"

# Shared experiment settings for Markov, LSTM, and the LLM editor.
START_LINE = "Я вышел ночью в сад пустой"
RHYME_SCHEME = "AABB"
POEM_LINES = 8
# Local LLM in Colab: free, no external API, requires GPU runtime.
os.environ["LLM_PROVIDER"] = "transformers"
os.environ.pop("OLLAMA_BASE_URL", None)
LLM_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

# Number of poems for the final comparison. Set to 20 for a stricter run.
EXPERIMENT_RUNS = 10
EXPERIMENT_LLM_RUNS = 3
MARKOV_GENERATION_RETRIES = 30
LSTM_DEMO_RETRIES = 2
LSTM_BATCH_RETRIES = 1

# Faster LSTM inference settings.
LSTM_FREE_LINE_CANDIDATES = 6
LSTM_RHYME_CANDIDATES = 8
LSTM_TOP_K = 25
LSTM_MAX_CONTEXT_TOKENS = 48

# Store the trained full LSTM model on Google Drive between Colab sessions.
USE_GOOGLE_DRIVE_CACHE = True
DRIVE_LSTM_CACHE_DIR = "/content/drive/MyDrive/poetry_lstm_cache"

# CSV report settings.
REPORTS_DIR = "/content/poetry/reports"
AUTO_DOWNLOAD_REPORTS = False

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("Settings ready. For local Mistral, use a GPU runtime in Colab.")


Settings ready


## 2. Установка библиотек

Для цепей Маркова нужна библиотека `markovify`. Для LSTM нужен TensorFlow, а для локального Mistral в Colab понадобятся `transformers`, `accelerate` и `bitsandbytes`.

In [4]:
import importlib.util
import subprocess
import sys


def install_if_missing(module_name, pip_name=None):
    pip_name = pip_name or module_name
    try:
        spec = importlib.util.find_spec(module_name)
    except ModuleNotFoundError:
        spec = None
    if spec is None:
        print(f"Устанавливаем {pip_name}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
    else:
        print(f"{module_name} уже установлен")


install_if_missing("markovify")
install_if_missing("tensorflow")
install_if_missing("transformers")
install_if_missing("accelerate")
install_if_missing("bitsandbytes")
install_if_missing("sentencepiece")

print("Библиотеки готовы")


markovify уже установлен
Устанавливаем tensorflow...


CalledProcessError: Command '['d:\\Python\\python.exe', '-m', 'pip', 'install', '-q', 'tensorflow']' returned non-zero exit status 1.

## 3. Загрузка проекта

Если папка `/content/poetry` уже есть, блокнот использует её. Если папки нет, проект будет скачан из GitHub.

In [ ]:
from pathlib import Path
import subprocess

project_path = Path(PROJECT_DIR)

if project_path.exists():
    print("Папка проекта уже есть:", PROJECT_DIR)
else:
    print("Скачиваем проект...")
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)

print("Содержимое проекта:")
for item in sorted(project_path.iterdir()):
    print("-", item.name)

## 4. Загрузка датасета

Для примера используется открытый датасет RIFMA. Мы скачиваем JSON и преобразуем его в простой текстовый файл `poems_clean.txt`, который нужен текущему проекту.

In [ ]:
import urllib.request

if Path(DATASET_JSON_PATH).exists():
    print("JSON-датасет уже скачан:", DATASET_JSON_PATH)
else:
    print("Скачиваем датасет...")
    urllib.request.urlretrieve(DATASET_JSON_URL, DATASET_JSON_PATH)

print("Файл датасета:", DATASET_JSON_PATH)
print("Размер файла:", Path(DATASET_JSON_PATH).stat().st_size, "байт")

## 5. Преобразование датасета в poems_clean.txt

Эта ячейка специально написана устойчиво: она ищет стихотворный текст даже если структура JSON немного отличается.

In [ ]:
import json
import re

with open(DATASET_JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Тип data:", type(data))
if hasattr(data, "__len__"):
    print("Размер data:", len(data))

if isinstance(data, dict):
    print("Ключи верхнего уровня:", list(data.keys())[:20])
    for key in ["data", "dataset", "items", "poems", "samples"]:
        if key in data and isinstance(data[key], list):
            data = data[key]
            print("Используем список из ключа:", key)
            break

if isinstance(data, list) and data:
    print("Пример первого элемента:")
    print(repr(data[0])[:1000])

def count_good_lines(text):
    count = 0
    for line in text.splitlines():
        words = re.findall(r"[А-Яа-яЁё]+", line)
        if len(words) >= 2:
            count += 1
    return count

def extract_text(item):
    candidates = []

    def walk(obj):
        if isinstance(obj, str):
            candidates.append(obj)
        elif isinstance(obj, dict):
            for value in obj.values():
                walk(value)
        elif isinstance(obj, (list, tuple)):
            for value in obj:
                walk(value)

    walk(item)

    if not candidates:
        return ""

    candidates.sort(
        key=lambda s: (count_good_lines(s), len(re.findall(r"[А-Яа-яЁё]", s))),
        reverse=True,
    )
    return candidates[0]

def remove_russian_accent_marks(text):
    # Убираем только знаки ударения. Букву й не трогаем.
    return text.replace("\u0301", "").replace("\u0300", "")

poems = []

for item in data:
    text = extract_text(item)
    if not text:
        continue

    text = remove_russian_accent_marks(text)

    clean_lines = []

    for line in text.splitlines():
        line = line.strip()
        line = re.sub(r"[^А-Яа-яЁё\-\s]", "", line)
        line = re.sub(r"\s+", " ", line).strip()

        words = re.findall(r"[А-Яа-яЁё]+", line)
        if len(words) >= 2:
            clean_lines.append(line)

    if len(clean_lines) >= 2:
        poems.append("\n".join(clean_lines))

with open(DATASET_TXT_PATH, "w", encoding="utf-8") as f:
    f.write("\n\n".join(poems))

print("Готово. Количество стихотворных фрагментов:", len(poems))

if len(poems) == 0:
    raise ValueError("Не удалось извлечь стихи из JSON. Нужно проверить структуру датасета.")

print("\nПервые строки poems_clean.txt:")
with open(DATASET_TXT_PATH, "r", encoding="utf-8") as f:
    for index, line in enumerate(f):
        if index >= 20:
            break
        print(line.rstrip())

## 6. Копирование датасета в папки проекта

In [ ]:
import shutil

markov_dataset_path = Path(PROJECT_DIR) / "Markov_chain_2" / "poems_clean.txt"
lstm_dataset_path = Path(PROJECT_DIR) / "LSTM_generation_2" / "poems_clean.txt"

shutil.copy(DATASET_TXT_PATH, markov_dataset_path)
shutil.copy(DATASET_TXT_PATH, lstm_dataset_path)

print("Датасет скопирован:")
print(markov_dataset_path)
print(lstm_dataset_path)

## 7. Быстрая проверка файлов проекта

Проверяем, что Python-файлы не содержат синтаксических ошибок.

In [ ]:
!python -m py_compile /content/poetry/Markov_chain_2/Markov_chain.py
!python -m py_compile /content/poetry/LSTM_generation_2/lstm_generation.py
!python -m py_compile /content/poetry/llm_poetry_tools.py

print("Синтаксис проверен")


## 8. Генерация через цепи Маркова

Эта часть обучается быстро.

In [ ]:
import sys

markov_dir = Path(PROJECT_DIR) / "Markov_chain_2"
sys.path.insert(0, str(markov_dir))
os.chdir(markov_dir)

from Markov_chain import RhymingPoetryGenerator


def clean_poem_lines(poem):
    if not poem:
        return []
    return [str(line).strip() for line in poem if str(line).strip()]


def generate_poem_with_retries(
    generator,
    lines=POEM_LINES,
    rhyme_scheme=RHYME_SCHEME,
    start_line=START_LINE,
    attempts=MARKOV_GENERATION_RETRIES,
    **kwargs,
):
    best_poem = []
    for attempt in range(1, attempts + 1):
        poem = generator.generate_poem(
            lines=lines,
            rhyme_scheme=rhyme_scheme,
            start_line=start_line,
            **kwargs,
        )
        poem_lines = clean_poem_lines(poem)
        if len(poem_lines) > len(best_poem):
            best_poem = poem_lines
        if len(poem_lines) >= lines:
            return poem_lines[:lines]
    if best_poem:
        print(
            f"Notice: got only {len(best_poem)} of {lines} lines "
            f"after {attempts} attempts."
        )
    return best_poem


markov_generator = RhymingPoetryGenerator(model_name="poet_markov")
markov_poem = None
markov_metrics = {}

if markov_generator.load_and_train("poems_clean.txt", state_size=2):
    markov_poem = generate_poem_with_retries(
        markov_generator,
        lines=POEM_LINES,
        rhyme_scheme=RHYME_SCHEME,
        start_line=START_LINE,
    )
    markov_generator.display_poem(markov_poem)
    markov_generator.display_metrics()
    markov_metrics = markov_generator.metrics.get_statistics()
else:
    print("Markov training failed. Check poems_clean.txt")


## 9. LSTM: быстрая проверка обучения

Сначала запускаем маленькую проверочную модель. Это нужно, чтобы убедиться, что обучение реально идёт.

Если видите строки `batch ...` и `Epoch ... Loss ...`, значит обучение работает.

In [ ]:
lstm_dir = Path(PROJECT_DIR) / "LSTM_generation_2"
sys.path.insert(0, str(lstm_dir))
os.chdir(lstm_dir)

from lstm_generation import LSTMRhymingPoetryGenerator

quick_lstm = LSTMRhymingPoetryGenerator(model_name="poet_quick")

quick_ok = quick_lstm.load_and_train(
    "poems_clean.txt",
    epochs=1,
    batch_size=16,
    max_vocab_size=3000,
    max_training_lines=2000,
)

if quick_ok:
    quick_poem = quick_lstm.generate_poem(
        lines=4,
        rhyme_scheme=RHYME_SCHEME,
        start_line=START_LINE,
        temperature=0.65,
    )
    quick_lstm.display_poem(quick_poem)
    quick_lstm.display_metrics()
else:
    print("LSTM не обучилась. Проверьте poems_clean.txt")


## LSTM cache on Google Drive

This optional block avoids retraining the full LSTM model on every Colab session. It restores `poet_model.keras`, `poet_metadata.json`, and `poet_rhymes.json` from Google Drive before training, then saves the current files back after training or loading.


In [ ]:
import shutil

_drive_cache_ready = False


def get_lstm_cache_files(model_name="poet"):
    return [
        f"{model_name}_model.keras",
        f"{model_name}_metadata.json",
        f"{model_name}_rhymes.json",
    ]


def prepare_drive_cache():
    global _drive_cache_ready
    if not USE_GOOGLE_DRIVE_CACHE:
        print("Google Drive cache is disabled.")
        return None
    if _drive_cache_ready:
        return Path(DRIVE_LSTM_CACHE_DIR)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print("Google Drive is unavailable; LSTM will use local files:", exc)
        return None

    cache_dir = Path(DRIVE_LSTM_CACHE_DIR)
    cache_dir.mkdir(parents=True, exist_ok=True)
    _drive_cache_ready = True
    print("LSTM cache directory:", cache_dir)
    return cache_dir


def restore_lstm_cache_from_drive(local_dir, model_name="poet"):
    cache_dir = prepare_drive_cache()
    if cache_dir is None:
        return False

    restored = []
    for filename in get_lstm_cache_files(model_name):
        src = cache_dir / filename
        dst = Path(local_dir) / filename
        if src.exists():
            shutil.copy2(src, dst)
            restored.append(filename)

    if restored:
        print("Restored LSTM files from Google Drive:", ", ".join(restored))
        return True

    print("No saved LSTM model found on Google Drive yet. It will be saved after training.")
    return False


def save_lstm_cache_to_drive(local_dir, model_name="poet"):
    cache_dir = prepare_drive_cache()
    if cache_dir is None:
        return False

    saved = []
    for filename in get_lstm_cache_files(model_name):
        src = Path(local_dir) / filename
        dst = cache_dir / filename
        if src.exists():
            shutil.copy2(src, dst)
            saved.append(filename)

    if saved:
        print("Saved LSTM files to Google Drive:", ", ".join(saved))
        return True

    print("No LSTM files were found for saving.")
    return False


## 10. LSTM: основной запуск

Запускайте эту ячейку после быстрой проверки. Она обучает модель дольше, использует больше слов и берёт стартовую строку как смысловой якорь.

LSTM теперь обучается на обычном порядке слов и видит границы строк через токен `<line>`. Для рифмы она генерирует несколько кандидатов и выбирает строку с более подходящим последним словом.

Если нужно быстрее, уменьшите `epochs` до `3`, `5` или `10`.

In [ ]:
full_lstm = LSTMRhymingPoetryGenerator(model_name="poet")
full_poem = None
full_lstm_metrics = {}

restore_lstm_cache_from_drive(lstm_dir, model_name="poet")

full_ok = full_lstm.load_and_train(
    "poems_clean.txt",
    epochs=20,
    batch_size=32,
    max_vocab_size=12000,
    max_training_lines=20000,
)

if full_ok:
    save_lstm_cache_to_drive(lstm_dir, model_name="poet")
    full_poem = generate_poem_with_retries(
        full_lstm,
        lines=POEM_LINES,
        rhyme_scheme=RHYME_SCHEME,
        start_line=START_LINE,
        attempts=LSTM_DEMO_RETRIES,
        temperature=0.65,
        free_line_candidates=LSTM_FREE_LINE_CANDIDATES,
        rhyme_candidates=LSTM_RHYME_CANDIDATES,
        top_k=LSTM_TOP_K,
        max_context_tokens=LSTM_MAX_CONTEXT_TOKENS,
    )
    full_lstm.display_poem(full_poem)
    full_lstm.display_metrics()
    full_lstm_metrics = full_lstm.metrics.get_statistics()
else:
    print("Full LSTM training failed. Check poems_clean.txt")


## 11. LLM-редактор и оценка осмысленности

Этот блок не заменяет Markov и LSTM. Сначала обе модели создают черновики, затем LLM используется как редактор и как автоматизированный эксперт для оценки смысловой связности.

По умолчанию этот блок запускает локальную Mistral-модель прямо в Colab через `transformers`, без внешнего API и без туннелей. Для этого нужен GPU runtime. Если локальная модель не загрузится, блоки LLM будут пропущены, а базовое сравнение Markov/LSTM останется рабочим.


In [ ]:
import sys

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Reload the helper module if it was already imported in this Colab runtime.
import importlib
import llm_poetry_tools
importlib.reload(llm_poetry_tools)

from llm_poetry_tools import (
    LLMPoetryAssistant,
    build_formal_metrics_table,
    build_poem_report_row,
    calculate_formal_metrics,
    export_report_bundle,
    print_poem,
)

llm = LLMPoetryAssistant(model_name=LLM_MODEL_NAME)
print(llm.status_message())

if not llm.is_available():
    print(
        "LLM blocks will be skipped. "
        "Check LLM_PROVIDER, install the required packages, and make sure Colab is using a GPU runtime."
    )


## 12. Редактирование черновиков через LLM

Здесь LLM получает только уже созданный черновик. В промпте фиксируются количество строк, первая строка, тема и схема рифмовки, чтобы модель работала именно как редактор.


In [ ]:
edited_markov_poem = None
edited_lstm_poem = None

poem_versions = {}
if markov_poem:
    poem_versions["Markov"] = markov_poem
if full_poem:
    poem_versions["LSTM"] = full_poem

if llm.is_available():
    if markov_poem:
        edited_markov_poem = llm.edit_poem(
            markov_poem,
            start_line=START_LINE,
            rhyme_scheme=RHYME_SCHEME,
        )
        poem_versions["Markov + LLM"] = edited_markov_poem
        print_poem("Markov + LLM-редактор", edited_markov_poem)

    if full_poem:
        edited_lstm_poem = llm.edit_poem(
            full_poem,
            start_line=START_LINE,
            rhyme_scheme=RHYME_SCHEME,
        )
        poem_versions["LSTM + LLM"] = edited_lstm_poem
        print_poem("LSTM + LLM-редактор", edited_lstm_poem)
else:
    print("Редактирование пропущено: LLM-модуль недоступен.")


## 13. Формальные метрики до и после LLM

Эта таблица нужна для проверки структуры: рифмы, уникальности, длины строк и словаря. Она не доказывает осмысленность, но показывает, сохранились ли формальные свойства стихотворения после редактирования.


In [ ]:
formal_rows = build_formal_metrics_table(poem_versions, rhyme_scheme=RHYME_SCHEME)

try:
    import pandas as pd
    formal_df = pd.DataFrame(formal_rows)
    display(formal_df)
except Exception:
    for row in formal_rows:
        print(row)


## 14. LLM-оценка осмысленности

Оценщик не получает название генератора в промпте. Он видит только текст стихотворения и первую строку, затем ставит оценки по фиксированной шкале от 1 до 5: смысловая связность, грамматика, сохранение темы, поэтичность и общее качество.


In [ ]:
llm_evaluation_rows = []

if llm.is_available() and poem_versions:
    llm_evaluation_rows = llm.evaluate_versions(
        poem_versions,
        start_line=START_LINE,
    )

    try:
        import pandas as pd
        llm_eval_df = pd.DataFrame(llm_evaluation_rows)
        display(llm_eval_df)
    except Exception:
        for row in llm_evaluation_rows:
            print(row)
else:
    print("LLM-оценка пропущена: нет доступного LLM-модуля или стихотворений.")


## Batch experiment: 10-20 poem generations

This section performs a series of generations instead of a single demo run. By default `EXPERIMENT_RUNS = 10`; set it to 20 in the first settings cell for a stricter comparison. Each run generates Markov and LSTM drafts and computes average formal metrics.


In [ ]:
batch_poem_records = []
batch_formal_rows = []

for run_index in range(1, EXPERIMENT_RUNS + 1):
    print(f"Generation {run_index}/{EXPERIMENT_RUNS}...")

    generated = {
        "Markov": generate_poem_with_retries(
            markov_generator,
            lines=POEM_LINES,
            rhyme_scheme=RHYME_SCHEME,
            start_line=START_LINE,
            attempts=MARKOV_GENERATION_RETRIES,
        ),
        "LSTM": generate_poem_with_retries(
            full_lstm,
            lines=POEM_LINES,
            rhyme_scheme=RHYME_SCHEME,
            start_line=START_LINE,
            attempts=LSTM_BATCH_RETRIES,
            temperature=0.65,
            free_line_candidates=LSTM_FREE_LINE_CANDIDATES,
            rhyme_candidates=LSTM_RHYME_CANDIDATES,
            top_k=LSTM_TOP_K,
            max_context_tokens=LSTM_MAX_CONTEXT_TOKENS,
        ) if full_ok else [],
    }

    for model_name, poem in generated.items():
        if not poem:
            continue
        batch_poem_records.append({
            "run": run_index,
            "model": model_name,
            "poem": poem,
        })
        row = {
            "run": run_index,
            "version": model_name,
        }
        row.update(calculate_formal_metrics(poem, rhyme_scheme=RHYME_SCHEME))
        batch_formal_rows.append(row)

try:
    import pandas as pd
    batch_formal_df = pd.DataFrame(batch_formal_rows)
    display(batch_formal_df)
    display(batch_formal_df.groupby("version").mean(numeric_only=True).round(2))
except Exception:
    for row in batch_formal_rows:
        print(row)


## LLM editing and evaluation for the batch

This cell applies the LLM to the generated series. The number of LLM-processed runs is controlled by `EXPERIMENT_LLM_RUNS`. It defaults to 10, but you can lower it if API quota or runtime is limited.


In [ ]:
batch_llm_rows = []
batch_llm_records = []

if llm.is_available() and batch_poem_records:
    records_for_llm = [
        record for record in batch_poem_records
        if record["run"] <= EXPERIMENT_LLM_RUNS
    ]

    for index, record in enumerate(records_for_llm, start=1):
        print(
            f"LLM processing {index}/{len(records_for_llm)}: "
            f"{record['model']} run {record['run']}"
        )
        edited = llm.edit_poem(
            record["poem"],
            start_line=START_LINE,
            rhyme_scheme=RHYME_SCHEME,
        )

        versions = {
            record["model"]: record["poem"],
            f"{record['model']} + LLM": edited,
        }

        for version_name, poem in versions.items():
            metrics = calculate_formal_metrics(poem, rhyme_scheme=RHYME_SCHEME)
            evaluation = llm.evaluate_poem(poem, start_line=START_LINE)
            row = {
                "run": record["run"],
                "version": version_name,
            }
            row.update(metrics)
            for field in [
                "semantic_coherence",
                "grammar",
                "theme_consistency",
                "poetic_quality",
                "overall",
            ]:
                row[field] = evaluation.get(field, 0)
            row["comment"] = evaluation.get("comment", "")
            batch_llm_rows.append(row)
            batch_llm_records.append(
                {
                    "run": record["run"],
                    "source_model": record["model"],
                    "version": version_name,
                    "stage": "llm_edited" if "+ LLM" in version_name else "raw",
                    "poem": poem,
                    "metrics": metrics,
                    "evaluation": evaluation,
                }
            )

    try:
        import pandas as pd
        batch_llm_df = pd.DataFrame(batch_llm_rows)
        display(batch_llm_df)
        score_columns = [
            "semantic_coherence",
            "grammar",
            "theme_consistency",
            "poetic_quality",
            "overall",
        ]
        display(batch_llm_df.groupby("version")[score_columns].mean().round(2))
    except Exception:
        for row in batch_llm_rows:
            print(row)
else:
    print("Batch LLM experiment skipped: LLM is unavailable or no poems were generated.")
    batch_llm_records = []


## CSV report export

This block builds a full CSV report for the whole cycle: demo poems, batch poems, raw drafts, edited versions, formal metrics, LLM scores, comments, and the full poem text split into separate line columns.


In [ ]:
report_rows = []

demo_formal_map = {row["name"]: row for row in formal_rows} if "formal_rows" in globals() else {}
demo_eval_map = {row["name"]: row for row in llm_evaluation_rows} if "llm_evaluation_rows" in globals() else {}
demo_versions = poem_versions if "poem_versions" in globals() else {}
batch_poems = batch_poem_records if "batch_poem_records" in globals() else []
batch_llm_data = batch_llm_records if "batch_llm_records" in globals() else []

for version_name, poem in demo_versions.items():
    report_rows.append(
        build_poem_report_row(
            version=version_name,
            poem=poem,
            report_scope="demo",
            run=1,
            start_line=START_LINE,
            rhyme_scheme=RHYME_SCHEME,
            requested_lines=POEM_LINES,
            llm_model_name=LLM_MODEL_NAME if llm.is_available() else "",
            metrics=demo_formal_map.get(version_name),
            evaluation=demo_eval_map.get(version_name),
            extra={
                "experiment_runs": EXPERIMENT_RUNS,
                "experiment_llm_runs": EXPERIMENT_LLM_RUNS,
                "markov_generation_retries": MARKOV_GENERATION_RETRIES,
                    "lstm_demo_retries": LSTM_DEMO_RETRIES,
                    "lstm_batch_retries": LSTM_BATCH_RETRIES,
            },
        )
    )

batch_eval_keys = {
    (record.get("run"), record.get("version"))
    for record in batch_llm_data
}

for record in batch_poems:
    key = (record.get("run"), record.get("model"))
    if key in batch_eval_keys:
        continue
    report_rows.append(
        build_poem_report_row(
            version=record["model"],
            poem=record["poem"],
            report_scope="batch",
            run=record["run"],
            source_model=record["model"],
            start_line=START_LINE,
            rhyme_scheme=RHYME_SCHEME,
            requested_lines=POEM_LINES,
            metrics=calculate_formal_metrics(record["poem"], rhyme_scheme=RHYME_SCHEME),
            extra={
                "experiment_runs": EXPERIMENT_RUNS,
                "experiment_llm_runs": EXPERIMENT_LLM_RUNS,
                "markov_generation_retries": MARKOV_GENERATION_RETRIES,
                    "lstm_demo_retries": LSTM_DEMO_RETRIES,
                    "lstm_batch_retries": LSTM_BATCH_RETRIES,
            },
        )
    )

for record in batch_llm_data:
    report_rows.append(
        build_poem_report_row(
            version=record["version"],
            poem=record["poem"],
            report_scope="batch",
            run=record["run"],
            source_model=record.get("source_model"),
            stage=record.get("stage"),
            start_line=START_LINE,
            rhyme_scheme=RHYME_SCHEME,
            requested_lines=POEM_LINES,
            llm_model_name=LLM_MODEL_NAME,
            metrics=record.get("metrics"),
            evaluation=record.get("evaluation"),
            extra={
                "experiment_runs": EXPERIMENT_RUNS,
                "experiment_llm_runs": EXPERIMENT_LLM_RUNS,
                "markov_generation_retries": MARKOV_GENERATION_RETRIES,
                    "lstm_demo_retries": LSTM_DEMO_RETRIES,
                    "lstm_batch_retries": LSTM_BATCH_RETRIES,
            },
        )
    )

report_bundle = export_report_bundle(
    report_rows,
    output_dir=REPORTS_DIR,
    prefix="poetry_report",
)

print("Detailed CSV:", report_bundle["detailed_path"])
print("Summary CSV:", report_bundle["summary_path"])

try:
    import pandas as pd
    display(pd.DataFrame(report_bundle["summary_rows"]))
except Exception:
    for row in report_bundle["summary_rows"]:
        print(row)


## Download CSV reports

Run this cell after the export step if you want the CSV files to download directly to your machine from Colab.


In [ ]:
if "report_bundle" not in globals():
    print("Run the CSV report export cell first.")
else:
    print("Detailed CSV:", report_bundle["detailed_path"])
    print("Summary CSV:", report_bundle["summary_path"])
    if AUTO_DOWNLOAD_REPORTS:
        try:
            from google.colab import files
            files.download(report_bundle["detailed_path"])
            files.download(report_bundle["summary_path"])
        except Exception as exc:
            print("Auto-download is unavailable in this environment:", exc)
    else:
        print(
            "AUTO_DOWNLOAD_REPORTS is False. "
            "Set it to True in the settings cell or download the files manually from the paths above."
        )


## 15. Что считать успешным результатом

Проект работает корректно, если:

- Markov-часть выводит статистику датасета, стихотворение и метрики.
- Быстрая LSTM-проверка выводит `batch ...`, затем `Epoch 1/1, Loss ...`.
- Основная LSTM-модель после обучения выводит стихотворение и метрики.
- При наличии `GEMINI_API_KEY` появляются версии `Markov + LLM` и `LSTM + LLM`, таблица формальных метрик и таблица LLM-оценки осмысленности.
- После запуска `CSV report export` сохраняются подробный и summary CSV-отчёты со всеми основными результатами.

Предупреждения TensorFlow про `cuFFT`, `cuDNN`, `cuBLAS` в Colab обычно не являются ошибкой.
